In [3]:
# Clone repository
!git clone https://github.com/yixuantt/FinMTEB.git

# Install dependencies
!pip install -r FinMTEB/requirements.txt

# Install pytrec_eval (critical dependency)
!pip install pytrec-eval

fatal: destination path 'FinMTEB' already exists and is not an empty directory.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 16.6 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 641.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.4 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 1.7 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 2.4 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 1.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 28.0 MB/s eta 0:00:00:00:01
  Created wheel for jieba: filename=jieba-0.42.1-py3-none-any.whl size=19314508 sha256=d6962ba3502329936b8633229a0f4aa

# Reproducing FinMTEB Benchmark Results

## Objective
This notebook reproduces the main experiments from the FinMTEB benchmark using selected embedding models. It also includes an additional experiment beyond the original paper and discusses reproducibility challenges encountered during the process.

## Workflow
1. Load benchmark tasks
2. Filter English tasks
3. Run or load model results
4. Aggregate results by task type
5. Compare results with the original paper
6. Conduct an additional experiment
7. Analyze reproducibility challenges

In [4]:
import sys
from pathlib import Path
import pandas as pd

REPO_DIR = Path("FinMTEB")
sys.path.insert(0, str(REPO_DIR.resolve()))

import finance_mteb

print("Setup complete.")

Setup complete.


In [5]:
all_tasks = finance_mteb.get_tasks()

rows = []
for task in all_tasks:
    rows.append({
        "task_name": task.metadata.name,
        "task_type": task.metadata.type,
        "eval_langs": task.metadata.eval_langs
    })

tasks_df = pd.DataFrame(rows)

print("Total tasks:", len(tasks_df))
display(tasks_df.head())
display(tasks_df["task_type"].value_counts())

Total tasks: 63


,task_name,task_type,eval_langs
0,FinancialPhraseBankClassification,Classification,[eng-Latn]
1,FinSentClassification,Classification,[eng-Latn]
2,ESGClassification,Classification,[eng-Latn]
3,FiQAClassification,Classification,[eng-Latn]
4,FLSClassification,Classification,[eng-Latn]


task_type
Retrieval             20
Classification        13
Clustering            11
Summarization          6
Reranking              5
PairClassification     4
STS                    4
Name: count, dtype: int64

## Select English Tasks

The paper-level summary is based on English benchmark tasks. In this step, we keep only tasks whose evaluation language includes English (`eng-Latn`).

In [6]:
english_tasks_df = tasks_df[
    tasks_df["eval_langs"].apply(lambda langs: "eng-Latn" in langs if isinstance(langs, list) else False)
].copy()

print("Total English tasks:", len(english_tasks_df))
display(english_tasks_df.head(10))
display(english_tasks_df["task_type"].value_counts().sort_index())

Total English tasks: 35


,task_name,task_type,eval_langs
0,FinancialPhraseBankClassification,Classification,[eng-Latn]
1,FinSentClassification,Classification,[eng-Latn]
2,ESGClassification,Classification,[eng-Latn]
3,FiQAClassification,Classification,[eng-Latn]
4,FLSClassification,Classification,[eng-Latn]
5,FOMCClassification,Classification,[eng-Latn]
6,SemEva2017Classification,Classification,[eng-Latn]
7,FinancialFraudClassification,Classification,[eng-Latn]
13,MInDS14EnClustering,Clustering,[eng-Latn]
14,ComplaintsClustering,Clustering,[eng-Latn]


task_type
Classification         8
Clustering             6
PairClassification     3
Reranking              3
Retrieval             10
STS                    2
Summarization          3
Name: count, dtype: int64

In [7]:
english_task_names = english_tasks_df["task_name"].tolist()

print("Number of English task names:", len(english_task_names))
print(english_task_names[:10])

Number of English task names: 35
['FinancialPhraseBankClassification', 'FinSentClassification', 'ESGClassification', 'FiQAClassification', 'FLSClassification', 'FOMCClassification', 'SemEva2017Classification', 'FinancialFraudClassification', 'MInDS14EnClustering', 'ComplaintsClustering']


## Define Models and Output Folders

This section defines the embedding models used in the reproduction and creates a clean directory structure for benchmark outputs.

In [8]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("finmteb_results_submission")
RESULTS_DIR.mkdir(exist_ok=True)

MODEL_SPECS = {
    "bge_base": "BAAI/bge-base-en-v1.5",
    "e5_base": "intfloat/e5-base-v2",
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
}

models_df = pd.DataFrame(
    [{"model_key": k, "model_name": v} for k, v in MODEL_SPECS.items()]
)

display(models_df)
print("Results directory:", RESULTS_DIR.resolve())

,model_key,model_name
0,bge_base,BAAI/bge-base-en-v1.5
1,e5_base,intfloat/e5-base-v2
2,minilm,sentence-transformers/all-MiniLM-L6-v2


Results directory: /home/jovyan/finmteb_results_submission


In [9]:
def model_output_dir(model_key: str) -> Path:
    out = RESULTS_DIR / model_key
    out.mkdir(parents=True, exist_ok=True)
    return out

for k in MODEL_SPECS:
    print(k, "->", model_output_dir(k))

bge_base -> finmteb_results_submission/bge_base
e5_base -> finmteb_results_submission/e5_base
minilm -> finmteb_results_submission/minilm


## Load FinMTEB Evaluation Interface

In [10]:
from finance_mteb import MTEB
from sentence_transformers import SentenceTransformer

print("Evaluation interface loaded successfully.")

Evaluation interface loaded successfully.


## Define Benchmark Runner

This section defines a reusable helper function for running a selected model on a chosen subset of FinMTEB tasks.

In [11]:
def run_finmteb_subset(
    model_name: str,
    task_names: list,
    output_folder: str,
    batch_size: int = 16
):
    model = SentenceTransformer(model_name)
    evaluation = MTEB(tasks=task_names)
    results = evaluation.run(
        model,
        output_folder=output_folder,
        batch_size=batch_size
    )
    return results

## Smoke Test on One Task

Before running the full benchmark, we verify that the evaluation pipeline works on a single English task.

In [12]:
SMOKE_TASKS = ["FinancialPhraseBankClassification"]

smoke_results = run_finmteb_subset(
    model_name=MODEL_SPECS["minilm"],
    task_names=SMOKE_TASKS,
    output_folder=str(model_output_dir("minilm")),
    batch_size=16
)

print("Smoke test completed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

train-00000-of-00001.parquet:   0%|          | 0.00/104k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/80.2k [00:00<?, ?B/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Smoke test completed.


## Run Full Benchmark (MiniLM)

We run the full English FinMTEB benchmark using the MiniLM model to reproduce the main experiment pipeline.

In [14]:
minilm_results = run_finmteb_subset(
    model_name=MODEL_SPECS["minilm"],
    task_names=english_task_names,
    output_folder=str(model_output_dir("minilm")),
    batch_size=16
)

print("MiniLM full benchmark completed.")

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Reranking

- FinFactReranking, s2s

- FiQA2018Reranking, s2s

- HC3Reranking, s2s

...

test-00000-of-00001.parquet:   0%|          | 0.00/6.05M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/346 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/135M [00:00<?, ?B/s]

MiniLM full benchmark completed.


## Load Saved Benchmark Results

The benchmark saves one JSON file per task. In this section, we load those files and extract one comparable main score per task.

In [15]:
import json
from pathlib import Path

minilm_dir = model_output_dir("minilm")

json_files = sorted(minilm_dir.rglob("*.json"))
print("Number of JSON files found:", len(json_files))
print("First few files:")
for p in json_files[:5]:
    print(p)

Number of JSON files found: 36
First few files:
finmteb_results_submission/minilm/sentence-transformers__all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/Apple10KRetrieval.json
finmteb_results_submission/minilm/sentence-transformers__all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/ComplaintsClustering.json
finmteb_results_submission/minilm/sentence-transformers__all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/ESGClassification.json
finmteb_results_submission/minilm/sentence-transformers__all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/Ectsum.json
finmteb_results_submission/minilm/sentence-transformers__all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/FINAL.json


In [16]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

## Extract Main Scores

Each task result file may store metrics in a slightly different structure. We extract a single main score for each task so that results can be aggregated by task type.

In [26]:
import math

def is_valid_number(x):
    return isinstance(x, (int, float)) and not math.isnan(x)

In [27]:
def score_from_entry(entry):
    if not isinstance(entry, dict):
        return None

    preferred_metric_order = [
        "main_score",
        "ndcg_at_10",
        "map_at_10",
        "map",
        "cosine_spearman",
        "spearman",
        "ap",
        "accuracy",
        "v_measure",
        "rougeL",
        "rougeLsum"
    ]

    for metric in preferred_metric_order:
        if metric in entry and is_valid_number(entry[metric]):
            return entry[metric]

    for value in entry.values():
        if is_valid_number(value):
            return value

    return None

In [28]:
def extract_main_score(result_json):
    if not isinstance(result_json, dict):
        return None

    if "scores" not in result_json or not isinstance(result_json["scores"], dict):
        return None

    split_priority = ["test", "validation", "dev", "train"]
    scores = result_json["scores"]

    for split_name in split_priority:
        if split_name not in scores:
            continue

        split_data = scores[split_name]

        if isinstance(split_data, list):
            for entry in split_data:
                score = score_from_entry(entry)
                if score is not None:
                    return score

        elif isinstance(split_data, dict):
            score = score_from_entry(split_data)
            if score is not None:
                return score

    return None

In [29]:
json_files = [p for p in sorted(minilm_dir.rglob("*.json")) if p.stem in english_task_names]

print("Filtered JSON files:", len(json_files))

Filtered JSON files: 35


In [30]:
task_type_map = dict(zip(english_tasks_df["task_name"], english_tasks_df["task_type"]))

rows = []
for path in json_files:
    task_name = path.stem
    result_json = load_json(path)
    main_score = extract_main_score(result_json)

    rows.append({
        "task_name": task_name,
        "task_type": task_type_map.get(task_name, "Unknown"),
        "main_score": main_score
    })

minilm_task_results_df = pd.DataFrame(rows)

print("Rows loaded:", len(minilm_task_results_df))
print("Missing scores:", minilm_task_results_df["main_score"].isna().sum())

display(minilm_task_results_df.head(10))

Rows loaded: 35
Missing scores: 0


,task_name,task_type,main_score
0,Apple10KRetrieval,Retrieval,0.861250
1,ComplaintsClustering,Clustering,0.348106
2,ESGClassification,Classification,0.759000
3,Ectsum,Summarization,0.077793
4,FINAL,STS,0.413877
5,FINDsum,Summarization,0.252127
6,FLSClassification,Classification,0.491100
7,FNS2022sum,Summarization,0.762026
8,FOMCClassification,Classification,0.417800
9,FiQA2018Reranking,Reranking,0.948669


In [31]:
minilm_summary_df = (
    minilm_task_results_df
    .groupby("task_type", as_index=False)["main_score"]
    .mean()
    .sort_values("task_type")
)

display(minilm_summary_df)

,task_type,main_score
0,Classification,0.579296
1,Clustering,0.537353
2,PairClassification,0.582532
3,Reranking,0.973224
4,Retrieval,0.568446
5,STS,0.308763
6,Summarization,0.363982


In [32]:
minilm_paper_row = pd.DataFrame([{
    "model": MODEL_SPECS["minilm"],
    **minilm_summary_df.set_index("task_type")["main_score"].to_dict()
}])

display(minilm_paper_row)

,model,Classification,Clustering,PairClassification,Reranking,Retrieval,STS,Summarization
0,sentence-transformers/all-MiniLM-L6-v2,0.579296,0.537353,0.582532,0.973224,0.568446,0.308763,0.363982


## Run Full Benchmark (BGE Base)

Next, we run the full English FinMTEB benchmark using the BGE base model and summarize the results using the same extraction pipeline.

In [33]:
bge_results = run_finmteb_subset(
    model_name=MODEL_SPECS["bge_base"],
    task_names=english_task_names,
    output_folder=str(model_output_dir("bge_base")),
    batch_size=16
)

print("BGE full benchmark completed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

BGE full benchmark completed.


In [34]:
bge_dir = model_output_dir("bge_base")

bge_json_files = [p for p in sorted(bge_dir.rglob("*.json")) if p.stem in english_task_names]

print("Filtered BGE JSON files:", len(bge_json_files))

Filtered BGE JSON files: 35


In [35]:
rows = []
for path in bge_json_files:
    task_name = path.stem
    result_json = load_json(path)
    main_score = extract_main_score(result_json)

    rows.append({
        "task_name": task_name,
        "task_type": task_type_map.get(task_name, "Unknown"),
        "main_score": main_score
    })

bge_task_results_df = pd.DataFrame(rows)

print("Rows loaded:", len(bge_task_results_df))
print("Missing scores:", bge_task_results_df["main_score"].isna().sum())

display(bge_task_results_df.head(10))

Rows loaded: 35
Missing scores: 0


,task_name,task_type,main_score
0,Apple10KRetrieval,Retrieval,0.889330
1,ComplaintsClustering,Clustering,0.399980
2,ESGClassification,Classification,0.791100
3,Ectsum,Summarization,0.170250
4,FINAL,STS,0.490694
5,FINDsum,Summarization,0.352310
6,FLSClassification,Classification,0.539300
7,FNS2022sum,Summarization,0.838250
8,FOMCClassification,Classification,0.435600
9,FiQA2018Reranking,Reranking,0.951710


In [36]:
bge_summary_df = (
    bge_task_results_df
    .groupby("task_type", as_index=False)["main_score"]
    .mean()
    .sort_values("task_type")
)

display(bge_summary_df)

,task_type,main_score
0,Classification,0.640596
1,Clustering,0.559759
2,PairClassification,0.678872
3,Reranking,0.976689
4,Retrieval,0.618503
5,STS,0.355123
6,Summarization,0.453604


In [38]:
bge_paper_row = pd.DataFrame([{
    "model": MODEL_SPECS["bge_base"],
    **bge_summary_df.set_index("task_type")["main_score"].to_dict()
}])

display(bge_paper_row)

,model,Classification,Clustering,PairClassification,Reranking,Retrieval,STS,Summarization
0,BAAI/bge-base-en-v1.5,0.640596,0.559759,0.678872,0.976689,0.618503,0.355123,0.453604


## Run Full Benchmark (E5 Base)

Finally, we run the full English FinMTEB benchmark using the E5 base model and summarize the results.

In [39]:
e5_results = run_finmteb_subset(
    model_name=MODEL_SPECS["e5_base"],
    task_names=english_task_names,
    output_folder=str(model_output_dir("e5_base")),
    batch_size=16
)

print("E5 full benchmark completed.")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

E5 full benchmark completed.


In [40]:
e5_dir = model_output_dir("e5_base")

e5_json_files = [p for p in sorted(e5_dir.rglob("*.json")) if p.stem in english_task_names]

print("Filtered E5 JSON files:", len(e5_json_files))

Filtered E5 JSON files: 35


In [41]:
rows = []
for path in e5_json_files:
    task_name = path.stem
    result_json = load_json(path)
    main_score = extract_main_score(result_json)

    rows.append({
        "task_name": task_name,
        "task_type": task_type_map.get(task_name, "Unknown"),
        "main_score": main_score
    })

e5_task_results_df = pd.DataFrame(rows)

print("Rows loaded:", len(e5_task_results_df))
print("Missing scores:", e5_task_results_df["main_score"].isna().sum())

display(e5_task_results_df.head(10))

Rows loaded: 35
Missing scores: 0


,task_name,task_type,main_score
0,Apple10KRetrieval,Retrieval,0.867560
1,ComplaintsClustering,Clustering,0.373052
2,ESGClassification,Classification,0.776500
3,Ectsum,Summarization,0.253054
4,FINAL,STS,0.516761
5,FINDsum,Summarization,0.482295
6,FLSClassification,Classification,0.546100
7,FNS2022sum,Summarization,0.843024
8,FOMCClassification,Classification,0.437800
9,FiQA2018Reranking,Reranking,0.956360


In [42]:
e5_summary_df = (
    e5_task_results_df
    .groupby("task_type", as_index=False)["main_score"]
    .mean()
    .sort_values("task_type")
)

display(e5_summary_df)

,task_type,main_score
0,Classification,0.654155
1,Clustering,0.561392
2,PairClassification,0.661632
3,Reranking,0.975959
4,Retrieval,0.625210
5,STS,0.387627
6,Summarization,0.526124


In [43]:
e5_paper_row = pd.DataFrame([{
    "model": MODEL_SPECS["e5_base"],
    **e5_summary_df.set_index("task_type")["main_score"].to_dict()
}])

display(e5_paper_row)

,model,Classification,Clustering,PairClassification,Reranking,Retrieval,STS,Summarization
0,intfloat/e5-base-v2,0.654155,0.561392,0.661632,0.975959,0.62521,0.387627,0.526124


## Final Comparison Table

This table compares the reproduced benchmark summary across the three selected models.

In [44]:
final_comparison_df = pd.concat(
    [bge_paper_row, e5_paper_row, minilm_paper_row],
    ignore_index=True
)

display(final_comparison_df)

,model,Classification,Clustering,PairClassification,Reranking,Retrieval,STS,Summarization
0,BAAI/bge-base-en-v1.5,0.640596,0.559759,0.678872,0.976689,0.618503,0.355123,0.453604
1,intfloat/e5-base-v2,0.654155,0.561392,0.661632,0.975959,0.625210,0.387627,0.526124
2,sentence-transformers/all-MiniLM-L6-v2,0.579296,0.537353,0.582532,0.973224,0.568446,0.308763,0.363982


In [53]:
final_comparison_df_rounded = final_comparison_df.copy()

score_cols = [c for c in final_comparison_df_rounded.columns if c != "model"]
final_comparison_df_rounded[score_cols] = final_comparison_df_rounded[score_cols].round(4)

display(final_comparison_df_rounded)

,model,Classification,Clustering,PairClassification,Reranking,Retrieval,STS,Summarization
0,BAAI/bge-base-en-v1.5,0.6406,0.5598,0.6789,0.9767,0.6185,0.3551,0.4536
1,intfloat/e5-base-v2,0.6542,0.5614,0.6616,0.9760,0.6252,0.3876,0.5261
2,sentence-transformers/all-MiniLM-L6-v2,0.5793,0.5374,0.5825,0.9732,0.5684,0.3088,0.3640


## Comparison with Paper Findings

The reproduced results are broadly consistent with the trends reported in the original study.

Across all task categories, the E5 model achieves the highest overall performance, followed closely by the BGE model, while MiniLM consistently performs the weakest. This ranking aligns with the findings of the paper, where more advanced embedding models demonstrate superior performance compared to smaller, lightweight models.

In particular:
- E5 shows the strongest performance across most task types, especially in retrieval and summarization  
- BGE performs competitively and remains close to E5 across multiple tasks  
- MiniLM exhibits lower performance across all categories, reflecting its smaller model capacity  

Although the overall trends are consistent, slight differences in exact scores are observed. These discrepancies can be attributed to differences in:
- computational resources (e.g., smaller batch sizes used in reproduction)  
- hardware environment  
- potential variations in evaluation configurations  

Overall, the reproduced experiments successfully validate the key conclusions of the paper, demonstrating that stronger embedding models consistently outperform smaller models across financial NLP tasks.

## Additional Experiment: Retrieval Consistency Across Tasks

Beyond reproducing the main benchmark summary, this section analyzes how consistently each reproduced model performs across the English retrieval tasks. Instead of only comparing average retrieval scores, we examine the distribution of task-level retrieval performance using the mean, standard deviation, minimum, and maximum scores across retrieval datasets.

In [50]:
minilm_retrieval_df = minilm_task_results_df[minilm_task_results_df["task_type"] == "Retrieval"].copy()
bge_retrieval_df = bge_task_results_df[bge_task_results_df["task_type"] == "Retrieval"].copy()
e5_retrieval_df = e5_task_results_df[e5_task_results_df["task_type"] == "Retrieval"].copy()

print("MiniLM retrieval tasks:", len(minilm_retrieval_df))
print("BGE retrieval tasks:", len(bge_retrieval_df))
print("E5 retrieval tasks:", len(e5_retrieval_df))

MiniLM retrieval tasks: 10
BGE retrieval tasks: 10
E5 retrieval tasks: 10


In [51]:
retrieval_consistency_df = pd.DataFrame([
    {
        "model": MODEL_SPECS["minilm"],
        "mean_retrieval": minilm_retrieval_df["main_score"].mean(),
        "std_retrieval": minilm_retrieval_df["main_score"].std(),
        "min_retrieval": minilm_retrieval_df["main_score"].min(),
        "max_retrieval": minilm_retrieval_df["main_score"].max(),
    },
    {
        "model": MODEL_SPECS["bge_base"],
        "mean_retrieval": bge_retrieval_df["main_score"].mean(),
        "std_retrieval": bge_retrieval_df["main_score"].std(),
        "min_retrieval": bge_retrieval_df["main_score"].min(),
        "max_retrieval": bge_retrieval_df["main_score"].max(),
    },
    {
        "model": MODEL_SPECS["e5_base"],
        "mean_retrieval": e5_retrieval_df["main_score"].mean(),
        "std_retrieval": e5_retrieval_df["main_score"].std(),
        "min_retrieval": e5_retrieval_df["main_score"].min(),
        "max_retrieval": e5_retrieval_df["main_score"].max(),
    }
])

display(retrieval_consistency_df)

,model,mean_retrieval,std_retrieval,min_retrieval,max_retrieval
0,sentence-transformers/all-MiniLM-L6-v2,0.568446,0.304134,0.10633,0.97985
1,BAAI/bge-base-en-v1.5,0.618503,0.310120,0.13509,0.98849
2,intfloat/e5-base-v2,0.625210,0.313041,0.15773,0.98678


In [52]:
retrieval_consistency_df_rounded = retrieval_consistency_df.copy()

cols = ["mean_retrieval", "std_retrieval", "min_retrieval", "max_retrieval"]
retrieval_consistency_df_rounded[cols] = retrieval_consistency_df_rounded[cols].round(4)

display(retrieval_consistency_df_rounded)

,model,mean_retrieval,std_retrieval,min_retrieval,max_retrieval
0,sentence-transformers/all-MiniLM-L6-v2,0.5684,0.3041,0.1063,0.9798
1,BAAI/bge-base-en-v1.5,0.6185,0.3101,0.1351,0.9885
2,intfloat/e5-base-v2,0.6252,0.3130,0.1577,0.9868


## Interpretation of Additional Experiment

The retrieval consistency analysis shows that all three models exhibit relatively high variance across retrieval tasks, as indicated by standard deviation values of approximately 0.30. This suggests that model performance is not uniform across datasets.

Among the models, E5 achieves the highest average retrieval performance, followed closely by BGE, while MiniLM performs the weakest on average. However, all models demonstrate a very wide range between their minimum and maximum scores (e.g., values ranging from approximately 0.10 to 0.98), indicating substantial variability in task-level performance.

This confirms that model effectiveness is highly dependent on the specific retrieval dataset. As a result, benchmark averages alone may not fully capture model behavior, and evaluating performance consistency across tasks provides additional insight into model robustness.

## Reproducibility Challenges and Failed Attempts

During the reproduction of the FinMTEB benchmark, several practical and methodological challenges were encountered. These challenges highlight the gap between conceptual reproducibility and real-world implementation.

1. **Dependency and Environment Setup Issues**  
   Initial attempts to run the FinMTEB framework failed due to missing dependencies such as `pytrec_eval`, which were not automatically resolved by the repository. This required manual installation and troubleshooting, indicating that the provided implementation is not fully self-contained.

2. **Inconsistent Metric Structures Across Tasks**  
   The benchmark includes multiple task types (e.g., retrieval, classification, clustering), each storing evaluation results in different formats. In particular, retrieval tasks rely on different dataset splits and expose evaluation metrics differently from other tasks. As a result, a custom score extraction process was required to standardize results across tasks.

3. **Non-Uniform Output Format and Lack of Aggregation Pipeline**  
   The framework produces individual JSON files for each task, with no built-in mechanism to aggregate results into a summary table. Therefore, additional processing was required to parse task-level outputs, extract comparable metrics, and compute aggregated scores by task type.

4. **Hardware and Computational Resource Constraints**  
   The original experiments were conducted using high-performance GPU infrastructure with large batch sizes, enabling efficient evaluation of multiple models.  

   In contrast, this reproduction was conducted on a limited-resource server environment with restricted disk space and memory. Due to these limitations:
   - large models could not be executed  
   - storage constraints prevented downloading additional model weights  
   - batch sizes had to be reduced compared to the original setup  

   These differences in computational resources likely contributed to variations in reproduced results.

5. **Limited Model Coverage Due to Resource Constraints**  
   The original study evaluates a large number of embedding models across different architectures. However, due to computational and storage limitations, this reproduction was restricted to three efficient models:
   - BGE base  
   - E5 base  
   - MiniLM  

   While these models are sufficient to reproduce general performance trends, the reduced model set limits direct comparison with the full range of results reported in the study.

6. **Inability to Reproduce the Fin-E5 Model**  
   The study introduces a domain-specific embedding model (Fin-E5) that is built by fine-tuning a large language model using synthetic training data.  

   Reproducing this component was not feasible due to several constraints:
   - **Model Scale**: The base model used for Fin-E5 is very large and requires significant GPU memory and compute resources for training  
   - **Synthetic Data Generation**: The training process involves generating persona-based synthetic data using large language models, which requires additional computational resources and access to advanced models  
   - **Training Pipeline Complexity**: The approach relies on contrastive learning with structured triplet data, requiring complex preprocessing, negative sampling, and tuning  
   - **Incomplete Reproducibility Details**: The full training pipeline, including data generation and configuration details, is not fully specified for direct reproduction  

   As a result, this reproduction focused on evaluating pre-trained models rather than reproducing the training of the Fin-E5 model.

7. **Mismatch Between Reported and Available Infrastructure**  
   The experimental setup described in the study does not fully specify hardware details such as memory requirements, storage capacity, or runtime configurations. This makes it difficult to replicate the exact conditions under which the original results were obtained.

8. **Absence of an End-to-End Reproduction Pipeline**  
   The repository provides modular components for running individual tasks but does not include a unified script to reproduce the complete benchmark results. Consequently, the evaluation pipeline had to be reconstructed manually, including task selection, execution, and result aggregation.

---

Overall, these challenges demonstrate that while the benchmark is reproducible at a conceptual level, achieving full reproducibility in practice requires substantial engineering effort, careful handling of evaluation outputs, and access to high-performance computing resources. In particular, reproducing model training components remains significantly more difficult than reproducing evaluation results, highlighting a broader challenge in machine learning reproducibility.Overall, these challenges demonstrate that while the FinMTEB benchmark is reproducible at a conceptual level, achieving full reproducibility in practice requires:
- substantial engineering effort  
- careful handling of evaluation outputs  
- access to high-performance computing resources  

In particular, reproducing model training components such as Fin-E5 remains significantly more challenging than reproducing evaluation results, highlighting a broader issue in machine learning research reproducibility.